# conv-leakyrelu-block-discriminator — worked example 1: Build a 64x64 to 32x32 discriminator downsampling block

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-leakyrelu-block-discriminator`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A DCGAN discriminator downsampling block is `Conv2d(kernel=4, stride=2, padding=1, bias=False) -> BatchNorm2d -> LeakyReLU(0.2)`. The stride-2 kernel-4 padding-1 convolution exactly halves each spatial dimension, and channels typically double. `bias=False` is used because the following BatchNorm carries its own learnable shift, making the conv bias redundant.

## Worked solution

**Step 1 — the conv.** We use `nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=False)`. The output spatial size obeys `floor((H + 2*pad - kernel)/stride) + 1 = floor((64 + 2 - 4)/2) + 1 = floor(62/2)+1 = 31+1 = 32`. So 64×64 becomes 32×32 — exactly halved. We disable bias because BatchNorm follows.

**Step 2 — BatchNorm.** `nn.BatchNorm2d(out_ch)` normalizes per-channel across the batch and spatial dims, stabilizing the discriminator's intermediate activations. Its affine `beta` parameter is what makes the conv bias redundant.

**Step 3 — LeakyReLU.** `nn.LeakyReLU(0.2, inplace=True)` is the DCGAN activation: a small negative slope (0.2) lets gradients flow for negative pre-activations, which empirically stabilizes GAN training compared to plain ReLU.

**Step 4 — wire and verify.** We pack the three layers into `nn.Sequential` in that exact order and push a `(2, 16, 64, 64)` tensor through it, expecting `(2, 32, 32, 32)`.

In [ ]:
import torch.nn as nn

def build_64_to_32_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.LeakyReLU(0.2, inplace=True),
    )

t.manual_seed(0)
block = build_64_to_32_block(16, 32)
x = t.randn(2, 16, 64, 64)
out = block(x)
print(tuple(out.shape))
print(block[0].bias is None)